# 📗 리랭킹으로 검색 후보를 다시 정렬합니다

## 리랭킹은 무엇인가요?

<strong>리랭킹(Reranking)은 1차 검색으로 찾은 후보 문서를 질문과 다시 비교해, 관련성이 높은 순서로 재정렬하는 방법입니다.</strong> 이 실습에서는 재정렬한 후보 중 답변에 사용할 상위 문서를 선택합니다.

검색 결과의 단어가 비슷해도 질문의 조건이나 예외에 답하지 못할 수 있습니다.

<img src="images/02_rerank_compress.png" width="1100" alt="1차 검색의 후보를 리랭킹해 문서를 선택하고, 컨텍스트 압축으로 질문에 필요한 내용을 추출하는 흐름">

그림에서 <strong>문서를 다시 정렬하고 선택하는 단계</strong>가 교안 01, <strong>선택한 문서의 내용을 줄이는 단계</strong>가 교안 02입니다. 그림의 문서 수는 흐름을 설명하기 위한 예시입니다.

## 이번 교안에서는 무엇을 하나요?

1. <strong>후보를 준비합니다.</strong> 지난 시간의 BM25·Dense 하이브리드 검색으로 최대 10개 후보를 찾습니다.
2. <strong>질문과 후보를 다시 비교합니다.</strong> Cross-Encoder(크로스인코더)에 질문과 문서를 함께 넣어 관련성 점수를 구합니다.
3. <strong>상위 문서를 선택하고 원문을 읽습니다.</strong> 최대 3개를 남긴 뒤, 이전 결과와 원문 ID·내용을 대조합니다.

리랭킹은 후보 안의 순서를 바꾸므로 <strong>후보에 없는 문서를 새로 찾지는 못합니다.</strong> 이 단계에서 선택한 문서의 본문은 그대로 유지합니다.

## TOP_K는 몇 개부터 시작하면 좋을까요?

<strong>일반적인 문서 질문·답변에서는 후보 20개를 찾고, 리랭킹 후 3~5개를 남기는 설정부터 시도해 볼 수 있습니다.</strong> 고정된 정답이 아니라, 아래 공식 문서의 예시를 참고한 시작값입니다. 문서 길이와 질문에 필요한 근거의 수에 맞춰 조정합니다.

먼저 <strong>검색할 후보 수</strong>와 <strong>최종 남길 문서 수</strong>를 구분하세요. 라이브러리마다 인자 이름은 다르지만, 이 교안에서는 다음처럼 읽습니다.

| 설정 | 의미 | 시작할 값 |
|---|---|---|
| 검색의 `k` 또는 `top_k` | 리랭커에 넘길 후보를 확보하는 수 | 전체 후보 20개부터 시작 |
| 리랭커의 `top_n` | 점수순으로 최종 선택할 문서 수 | 3~5개부터 시작 |
| 이 교안의 작은 실습 | BM25 5개 + Dense 5개를 중복 제거한 뒤 선택 | 후보 최대 10개 → 최종 3개 |

<strong>어떻게 조정할까요?</strong>

- 필요한 근거가 후보에 없으면 후보 수를 <strong>20 → 50 → 100개</strong>로 넓혀 봅니다. 리랭킹은 후보에 없는 문서를 가져오지 못합니다.
- 후보에는 근거가 있는데 최종 결과에서 빠졌다면 <strong>점수·원문을 확인하고 `top_n`을 3 → 5개로</strong> 늘려 봅니다. 여러 문서의 근거가 필요한 질문은 더 많은 문서가 필요할 수 있습니다.
- 원문이 길거나 중복·무관한 내용이 많으면 최종 개수를 줄이거나 청크 크기를 조정합니다. <strong>문서 개수뿐 아니라 LLM에 전달할 총 토큰 수</strong>도 함께 봅니다.

위의 ‘후보 20개’는 <strong>중복을 제거하고 리랭커에 실제로 넘기는 전체 후보 수</strong>를 뜻합니다. 하이브리드 검색에서 BM25·Dense를 각각 `k=20`으로 설정하면 합집합은 최대 40개가 될 수 있습니다. 이 교안은 각 `k=5`이므로 최대 10개이며, 중복이 있으면 그보다 적게 나옵니다.

설정 근거: [Pinecone 공식 예시](https://sdk.pinecone.io/python/how-to/inference/reranking.html#reranking-in-a-pipeline)는 후보 20개 → 최종 5개를 사용하고, [작은 값부터 조정하는 안내](https://www.pinecone.io/learn/refine-with-rerank/)는 10개 → 3개를 시작 예로 듭니다. [Sentence Transformers 예시](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html)는 후보 100개를 재정렬하는 흐름도 보여 줍니다. 서로 다른 예시가 있으므로 한 숫자를 모든 데이터에 고정하지 않습니다.

## Advanced RAG에서 이번 단원의 위치

지난 시간에는 질문을 바꾸고 여러 검색 결과를 합쳤습니다. 이번 단원은 <strong>검색 후(Post-retrieval)</strong> 단계에서 LLM에 전달할 근거를 다룹니다.

| 단계 | 하는 일 | 학습 위치 |
|---|---|---|
| 검색 전 | 검색에 넣을 질문을 바꿉니다. | day47 질의 변환 |
| 검색 | BM25·Dense의 결과를 합쳐 후보를 찾습니다. | day47 하이브리드 검색 |
| 검색 후 | 후보를 다시 정렬하고 필요한 내용을 추출합니다. | <strong>day48 리랭킹·컨텍스트 압축</strong> |

## 오늘의 목표

- [ ] 1차 후보 개수와 최종 선택 개수를 구분할 수 있습니다.
- [ ] Bi-Encoder·Cross-Encoder·ColBERT의 비교 방식을 설명할 수 있습니다.
- [ ] 같은 후보를 Cross-Encoder와 ColBERT 방식으로 재정렬하고 원문 ID로 비교할 수 있습니다.

시연은 <strong>수업용 가상 도서 51권의 소개·학습 내용</strong>에서 에이전트 구축을 배울 책을 찾습니다. 따라하기는 앞 단원과 같은 <strong>고용노동부 매뉴얼 44절</strong>을 사용합니다. 책 제목과 소개는 학습을 위해 만든 자료이며 실제 출판물 정보가 아닙니다.


## 모델 파일을 미리 준비합니다

패키지를 설치한 뒤 아래 셀을 먼저 실행합니다. Qwen은 약 1.19GB, BGE-M3는 약 2.3GB의 가중치를 받습니다. 이 단계는 파일만 저장하며 모델 추론·OpenAI API는 호출하지 않습니다. 이후 실습에서 같은 모델 ID를 사용하면 캐시를 재사용합니다.

[Hugging Face 다운로드 공식 안내](https://huggingface.co/docs/huggingface_hub/guides/download)


In [ ]:
# ====== Hugging Face 모델 파일 미리 다운로드 ======
from huggingface_hub import snapshot_download

# Qwen은 Cross-Encoder, BGE-M3는 ColBERT 방식에 사용합니다.
model_ids = [
    "Qwen/Qwen3-Reranker-0.6B",
    "BAAI/bge-m3",
]
for model_id in model_ids:
    cache_path = snapshot_download(
        repo_id=model_id,
        # 모델 가중치·토크나이저·설정만 받고 ONNX 변환본과 이미지는 제외합니다.
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.pt", "*.model", "*.txt", "*.jinja"],
        ignore_patterns=["onnx/*", "openvino/*"],
    )
    print("다운로드 완료:", model_id)
    print("캐시 위치:", cache_path)


## 준비와 데이터 살펴보기

노트북이 있는 폴더에서 새 커널로 시작하세요. 경로·JSON 함수·원문 ID는 지난 교안의 방식을 이어갑니다.

따라하기는 책 소개와 다른 자료인 실제 고용노동부 매뉴얼 44절을 사용합니다. 본문과 PDF 위치는 지난 자료 그대로입니다.

아래 준비 코드는 한 셀입니다. `# ====` 구분선으로 역할을 나눴습니다. 위에서부터 실행하면 유틸리티와 시연·따라하기용 검색기가 준비됩니다. 이 셀을 다시 실행하면 문서를 다시 임베딩합니다.


In [ ]:
# ====================================================================
# 1) 라이브러리 가져오기
# 문서·표·토큰화·검색에 사용할 라이브러리를 가져옵니다.
# ====================================================================
# 본문과 출처를 같은 Document에 보관합니다.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from kiwipiepy import Kiwi
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# ====================================================================
# 2) 경로와 JSON 입출력
# material_dir를 기준으로 데이터를 읽고 결과를 저장하는 함수를 준비합니다.
# ====================================================================
# 실행할 노트북 폴더를 기준으로 입력 data와 생성 결과 output을 구분합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )

# ====================================================================
# 3) 환경변수와 모델 설정
# .env의 API 키를 읽고 문서와 질문에 공통으로 사용할 모델을 설정합니다.
# ====================================================================
# 현재 작업 폴더부터 상위로 .env를 찾아 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))

# 문서와 질문을 같은 모델·차원으로 바꿔야 같은 인덱스에서 거리를 비교할 수 있습니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,
    check_embedding_ctx_length=False,  # 자동 길이 검사·분할을 끄고 준비한 짧은 문서를 보냅니다.
)
print("모델 연결 설정 완료. 임베딩은 아래 적재·검색 구간에서 요청합니다.")

# ====================================================================
# 4) 원문을 Document로 변환하는 함수
# 본문과 원문 ID·출처·검색 조건을 함께 보관합니다.
# ====================================================================
def make_documents(records):
    """원문 기록의 본문·출처와 검색 조건을 LangChain Document로 바꿉니다."""
    # id는 저장소가, metadata의 source_id는 RRF가 원문을 구별할 때 읽습니다.
    # 두 위치에 같은 원문 ID를 유지하고 필터 필드만 추가합니다.
    documents = []
    for record in records:
        document = Document(
            id=record["doc_id"],
            page_content=record["text"],
            metadata={"source_id": record["doc_id"], "title": record["title"],
                      "url": record["url"], **record["metadata"]},
        )
        documents.append(document)
    return documents

# ====================================================================
# 5) 시연 데이터 읽기
# 가상 도서 51권의 소개·학습 내용을 읽고 Document 목록으로 바꿉니다.
# ====================================================================
# 데이터 구조는 앞 5행으로 확인하고, 검색에는 전체 문서를 사용합니다.
records = read_json("demo_docs.json")
print("전체 문서 수:", len(records))
display(pd.DataFrame(records).head())
documents = make_documents(records)

# ====================================================================
# 6) 결과 출력과 한국어 토큰화
# show_results는 원문을 print로 보여 주고, kiwi_tokenize는 BM25에 쓸 토큰을 만듭니다.
# ====================================================================
def show_results(documents):
    """표시 순서와 원문 ID·메타데이터·본문 전체를 보여 줍니다."""
    # 합집합·조건 목록에서도 쓰므로 번호를 관련성 순위라고 부르지 않습니다.
    print("결과 문서 수:", len(documents))
    for order, doc in enumerate(documents, start=1):
        print(f"[{order}] 원문 ID:", doc.metadata["source_id"])
        print("메타데이터:", doc.metadata)
        # 본문은 별도 줄에 출력해 원문의 줄바꿈을 그대로 읽습니다.
        print("본문:")
        print(doc.page_content)
        print()

# 분석기는 한 번 준비해 문서와 질문 양쪽에 같은 방식으로 적용합니다.
kiwi = Kiwi()


def kiwi_tokenize(text):
    """명사·외국어·숫자를 소문자 토큰 목록으로 돌려줍니다."""
    # PDF에서 온 반각 가운뎃점(･)은 분석기가 앞뒤를 다른 낱말로 끊으므로 가운뎃점(·)으로 바꿉니다.
    text = text.replace("･", "·")
    # N 계열은 명사, SL은 외국어, SN은 숫자입니다. 조사는 제외합니다.
    # lower는 영문 대소문자를 통일합니다. 기호가 중요한 제품 코드는 별도로 살펴봅니다.
    words = []
    for token in kiwi.tokenize(text):
        if token.tag.startswith("N") or token.tag in {"SL", "SN"}:
            words.append(token.form.lower())
    return words

# ====================================================================
# 7) 시연용 하이브리드 검색기
# 문서를 임베딩해 Chroma에 적재하고 BM25·Dense 검색기를 RRF로 연결합니다.
# ====================================================================
# 같은 이름의 수업용 메모리 컬렉션만 비웁니다. 재실행하면 문서를 다시 임베딩합니다.
vector_store = Chroma(collection_name="day48_lesson01_books", embedding_function=embedding_model)
vector_store.reset_collection()
# 같은 원문 ID를 저장소 ID와 검색 결과 메타데이터에 함께 유지합니다.
added_ids = vector_store.add_documents(documents, ids=[doc.metadata["source_id"] for doc in documents])
print("day48_lesson01_books 적재 수:", len(added_ids))

# 각 검색기가 최대 5개씩 찾으므로 합친 후보는 최대 10개입니다. 리랭킹으로 3개를 선택합니다.
bm25 = BM25Retriever.from_documents(documents, preprocess_func=kiwi_tokenize, k=5)
dense = vector_store.as_retriever(search_kwargs={"k": 5})
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5], c=60, id_key="source_id")

# ====================================================================
# 8) 따라하기용 PDF 데이터와 검색기
# 매뉴얼 44절을 별도 저장소에 적재하고 practice_hybrid를 준비합니다.
# ====================================================================
# PDF 자료도 앞 5행만 살펴보고 전체 문서를 검색에 사용합니다.
practice_records = read_json("practice_docs.json")
print("따라하기 전체 문서 수:", len(practice_records))
display(pd.DataFrame(practice_records).head())
practice_documents = make_documents(practice_records)

# 같은 이름의 수업용 메모리 컬렉션만 비웁니다. 재실행하면 문서를 다시 임베딩합니다.
practice_store = Chroma(collection_name="day48_lesson01_books_hr", embedding_function=embedding_model)
practice_store.reset_collection()
# 같은 원문 ID를 저장소 ID와 검색 결과 메타데이터에 함께 유지합니다.
added_ids = practice_store.add_documents(practice_documents, ids=[doc.metadata["source_id"] for doc in practice_documents])
print("day48_lesson01_books_hr 적재 수:", len(added_ids))

# 원리 비교 동안 같은 검색 단위와 후보 설정을 유지합니다.
practice_bm25 = BM25Retriever.from_documents(practice_documents, preprocess_func=kiwi_tokenize, k=5)
practice_dense = practice_store.as_retriever(search_kwargs={"k": 5})
practice_hybrid = EnsembleRetriever(
    retrievers=[practice_bm25, practice_dense], weights=[0.5, 0.5], c=60, id_key="source_id",
)


## 1. 후보와 최종 결과를 구분합니다

검색에서 준비할 후보 수와 리랭킹 후 남길 문서 수를 따로 정합니다.

각 검색기의 `k=5`이면 BM25·Dense의 합집합은 최대 10개입니다. 후보를 넓게 잡으면 놓친 근거를 포함할 가능성이 커지지만, 후보마다 처리 시간도 늘어납니다. 최종 `top_n=3`는 그중 읽을 문서 수입니다. 후보 최대 10개와 최종 3개는 서로 다른 설정입니다.

<img src="images/lesson01_section01_candidates.png" width="1100" alt="BM25와 Dense의 후보를 합치고 중복을 제거한 뒤, 리랭킹으로 최종 문서를 선택합니다.">

BM25와 Dense의 후보를 합치고 중복을 제거한 뒤, 리랭킹으로 최종 문서를 선택합니다.


In [ ]:
# 전후 비교에 사용할 후보는 한 번 검색한 뒤 보관합니다.
question = "에이전트가 도구를 잘못 고르거나 호출에 실패할 때 실행 기록을 추적해 원인을 찾는 방법을 배울 책을 찾아 주세요."
candidates = hybrid.invoke(question)
# 리랭킹 결과(top_n=3)와 같은 개수로 비교할 상위 3개를 보관합니다.
initial_top = candidates[:3]

print("후보 수:", len(candidates), "비교할 최종 개수:", 3)
show_results(candidates)


이 질문의 조건은 두 가지입니다. <strong>언어 모델 에이전트가 도구를 잘못 고르거나 호출에 실패하는 상황</strong>을 다루고, <strong>실행 기록을 따라가 그 원인을 찾는 방법</strong>까지 설명해야 합니다.

후보 원문을 읽으며 두 조건을 모두 다루는 책과 한쪽만 맞는 책을 구분해 두세요. 책 소개의 마지막 문장은 대부분 그 책이 다루지 않는 범위입니다. 이 구분이 뒤의 리랭킹 전후 비교의 기준이 됩니다.


### 🖐️ 함께 따라하기: PDF에서 재택근무 제한 조건의 후보를 찾습니다

질문에 맞는 PDF 후보를 검색하고 비교용 상위 3개를 보관합니다. 작성 셀의 `...`만 채운 뒤 제공된 출력 셀을 실행하세요. 고객사 미팅이 있을 때 재택근무를 제한하는 조건과 일부 요일·시간만 활용하는 대안이 후보 어디에 있는지 확인하세요.


#### 1) 후보 검색과 상위 문서 보관

`practice_candidates`와 `practice_initial_top`을 만드는 두 부분을 채우세요.


In [ ]:
# [작성] ... 부분에 핵심 코드를 작성하세요.
# 이후 리랭킹에서도 이 후보를 재사용해 비교 기준을 유지합니다.
practice_question = "고객사와 정기 회의가 있는 날에도 재택근무를 할 수 있나요?"
practice_candidates = ...
# 리랭킹 전후를 같은 개수로 비교하기 위해 상위 3개를 보관합니다.
practice_initial_top = ...


#### 2) 전체 후보와 상위 문서 확인

제공 셀을 실행해 전체 후보 수와 비교용 3개를 구분하세요.


In [ ]:
# [제공 코드]

# 두 목록의 크기와 원문을 확인합니다.
print("전체 후보 수:", len(practice_candidates))
print("비교용 상위 문서 수:", len(practice_initial_top))
show_results(practice_candidates)


## 2. 질문과 문서를 비교하는 방식을 살펴봅니다

<strong>Bi-Encoder(바이인코더)</strong>는 질문과 문서를 따로 벡터로 만들고 유사도를 비교합니다. 지금까지 사용한 <strong>의미 기반 검색(Dense)</strong>이 이 방식입니다. 문서 벡터를 미리 저장해 두고 새 질문의 벡터와 유사도를 비교합니다.

<strong>Cross-Encoder(크로스인코더)</strong>는 질문과 후보 문서를 한 쌍으로 모델에 넣어 관련성 점수를 얻습니다. 두 텍스트의 관계를 함께 볼 수 있지만 질문이 바뀌면 후보마다 다시 처리합니다.

<strong>ColBERT</strong>는 질문과 문서를 각각 인코딩하되, 하나의 벡터로 합치지 않고 <strong>토큰별 벡터</strong>를 남깁니다. 그다음 비교하는 방식을 늦은 상호작용(late interaction)이라고 합니다.

<img src="images/lesson01_section02_scores.png" width="1100" alt="Cross-Encoder 점수는 입력한 질문·문서 쌍의 순서대로 반환됩니다. 정렬은 점수를 받은 뒤 수행합니다.">

Cross-Encoder 점수는 입력한 질문·문서 쌍의 순서대로 반환됩니다. 정렬은 점수를 받은 뒤 수행합니다.


ColBERT의 <strong>MaxSim</strong>은 ‘질문의 각 토큰마다 가장 잘 맞는 문서 토큰을 찾고, 그 유사도를 모두 더한다’는 뜻입니다. 이름의 Max는 각 질문 토큰에 대해 문서 쪽 최대 유사도를 고른다는 뜻입니다. 이렇게 고른 값을 질문 토큰별로 합산하므로 질문 전체에서 최고값 하나만 고르거나 모든 토큰을 평균하는 방식이 아닙니다. 문서의 토큰별 벡터를 미리 준비할 수 있지만 저장량은 늘어납니다. 전체 문서 검색에도, 후보 재정렬에도 사용할 수 있습니다. [ColBERT 원논문](https://arxiv.org/abs/2004.12832)

| 방식 | 문서 쪽에 미리 준비하는 것 | 질문이 들어온 뒤 하는 일 |
|---|---|---|
| 기존 의미 기반 검색(Dense, Bi-Encoder) | 문서마다 하나의 벡터 | 질문을 벡터로 바꾸고 문서 벡터와 유사도 비교 |
| Cross-Encoder | 원문 | 질문·후보를 함께 읽고 점수 계산 |
| ColBERT | 문서의 토큰별 벡터 | 질문 토큰마다 가장 유사한 문서 토큰을 찾아 점수 합산 |

어떤 방식이 항상 더 정확하거나 빠르다고 단정하지 않습니다. 언어·문서 길이·모델·후보 수에 따라 달라집니다.

### Cross-Encoder 실습: 질문·문서 쌍의 관련성 점수 계산

아래에서는 Qwen3-Reranker-0.6B에 질문과 후보 문서를 한 쌍씩 넣고 관련성 점수를 확인합니다. 약 6억 파라미터의 다국어 모델로, 모델 크기와 라이브러리 연결 편의를 고려한 기본 실습 모델입니다. 이 모델로 점수 계산과 상위 문서 선택을 연습합니다.


In [ ]:
# 리랭커 인터페이스를 사용합니다.
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker


기본 모델은 다국어 `Qwen/Qwen3-Reranker-0.6B`입니다. 최초 모델 가중치 다운로드는 약 1.19GB이며 실행 메모리는 별도로 필요합니다. 아래는 CPU에서 처리하고 질문·문서 입력을 1024토큰으로 제한합니다. 긴 입력은 뒤쪽 내용이 잘릴 수 있으므로 본문의 길이도 확인하세요.


In [ ]:
# 첫 실행에는 공개 모델 파일을 내려받습니다. 노트북마다 한 번만 준비합니다.
# 실습은 CPU와 입력 상한 1024토큰을 사용합니다. 긴 입력은 잘릴 수 있습니다.
cross_encoder = HuggingFaceCrossEncoder(
    model_name="Qwen/Qwen3-Reranker-0.6B",
    model_kwargs={"device": "cpu", "max_length": 1024},
)


#### 1) 모델 점수를 계산합니다

후보 전체를 한 번 점수화하고 반환된 점수 목록을 확인합니다.


In [ ]:
question = "에이전트가 도구를 잘못 고르거나 호출에 실패할 때 실행 기록을 추적해 원인을 찾는 방법을 배울 책을 찾아 주세요."

# 점수는 입력한 (질문, 문서) 쌍과 같은 순서로 대응합니다.
pairs = []
for doc in candidates:
    pairs.append((question, doc.page_content))
scores = cross_encoder.score(pairs)

print("모델이 반환한 점수(후보 순서):", scores)


#### 2) 순위 비교표를 만듭니다

원래 후보 순위를 보관하고, 점수순으로 정렬한 표에 새 순위를 붙입니다.


In [ ]:
# zip으로 후보와 점수를 묶고, enumerate로 검색 당시 순위를 1부터 붙입니다.
score_rows = []
for before_rank, (doc, score) in enumerate(zip(candidates, scores), start=1):
    score_rows.append({
        "before_rank": before_rank,
        "question": question, "source_id": doc.metadata["source_id"],
        "title": doc.metadata["title"], "text": doc.page_content, "score": float(score),
    })
# 검색 당시 순위는 유지하고, 별도 표를 점수순으로 정렬합니다.
score_table = pd.DataFrame(score_rows)
# 점수가 같으면 검색 당시 순서를 유지합니다.
ranked_score_table = score_table.sort_values("score", ascending=False, kind="stable")
ranked_score_table = ranked_score_table.reset_index(drop=True)
ranked_score_table["after_rank"] = range(1, len(ranked_score_table) + 1)


#### 3) 두 순서를 출력해 비교합니다

두 표의 원문 ID와 순위를 비교합니다. 출력만 다시 볼 때는 이 셀만 실행하세요.


In [ ]:
# 저장된 표를 출력하므로 이 셀은 모델을 다시 호출하지 않습니다.
print("검색 당시 순서(before_rank)")
display(score_table[["before_rank", "source_id", "title", "score"]])
print("점수에 따른 순서(after_rank): before_rank와 비교하세요.")
# 같은 행의 두 순위와 원문 ID를 보고 어떤 문서가 이동했는지 확인합니다.
display(ranked_score_table[["before_rank", "after_rank", "source_id", "title", "score"]])


높은 점수는 이 모델이 이 질문에 더 관련 있다고 판단했다는 뜻입니다. 점수를 정답 확률로 읽거나 Dense 유사도·다른 리랭커 점수와 더하지 않습니다. `before_rank`는 하이브리드 검색이 반환한 후보 순위이고, `after_rank`는 같은 후보 전체를 점수순으로 정렬한 뒤의 순위입니다. 첫 표에서 검색 당시 순서를 보고, 둘째 표에서 같은 문서의 두 순위와 점수를 비교하세요. 점수가 같으면 검색 당시 순서를 유지합니다. 표의 ID에 해당하는 본문이 실제로 질문에 답하는지 읽습니다.


### 🖐️ 함께 따라하기: Cross-Encoder로 PDF 후보의 점수를 구합니다

PDF 후보의 관련성 점수를 계산하는 핵심 호출을 작성합니다. 점수 출력과 순위 비교표 코드는 제공되어 있습니다. 어떤 문서가 올라가거나 내려갔는지 원문과 함께 확인하세요.


#### 1) 후보별 관련성 점수 계산

질문·문서 쌍을 만드는 반복문은 제공되어 있습니다. `practice_scores`에 점수를 받는 호출을 작성하세요.


In [ ]:
# [작성] ... 부분에 핵심 코드를 작성하세요.
practice_question = "고객사와 정기 회의가 있는 날에도 재택근무를 할 수 있나요?"

# 질문·후보·점수의 대응을 유지해야 점수를 다른 문서에 붙이지 않습니다.
practice_pairs = []
for doc in practice_candidates:
    practice_pairs.append((practice_question, doc.page_content))
practice_scores = ...


#### 2) 점수와 비교표 준비

제공된 코드는 원래 후보 순위와 점수순 순위를 표에 담습니다. 직접 다시 작성하지 않아도 됩니다.


In [ ]:
# [제공 코드]

print("모델이 반환한 점수(후보 순서):", practice_scores)

# 같은 위치의 PDF 후보와 점수를 묶고 검색 당시 순위를 함께 기록합니다.
practice_score_rows = []
for before_rank, (doc, score) in enumerate(zip(practice_candidates, practice_scores), start=1):
    practice_score_rows.append({
        "before_rank": before_rank,
        "question": practice_question, "source_id": doc.metadata["source_id"],
        "title": doc.metadata["title"], "text": doc.page_content, "score": float(score),
    })
# 검색 당시 순위는 유지하고, 별도 표를 점수순으로 정렬합니다.
practice_score_table = pd.DataFrame(practice_score_rows)
# 점수가 같으면 검색 당시 순서를 유지합니다.
practice_ranked_score_table = practice_score_table.sort_values("score", ascending=False, kind="stable")
practice_ranked_score_table = practice_ranked_score_table.reset_index(drop=True)
practice_ranked_score_table["after_rank"] = range(1, len(practice_ranked_score_table) + 1)


#### 3) 두 순서 비교

제공 셀을 실행해 `before_rank`, `after_rank`, `score`와 본문을 비교하세요.


In [ ]:
# [제공 코드]

# 저장된 표를 출력하므로 이 셀은 모델을 다시 호출하지 않습니다.
print("검색 당시 순서(before_rank)")
display(practice_score_table[["before_rank", "source_id", "title", "score"]])
print("점수에 따른 순서(after_rank): before_rank와 비교하세요.")
# 같은 행의 두 순위와 원문 ID를 보고 어떤 문서가 이동했는지 확인합니다.
display(practice_ranked_score_table[["before_rank", "after_rank", "source_id", "title", "score"]])


## 3. CrossEncoderReranker로 상위 문서를 선택합니다

`CrossEncoderReranker`는 <strong>점수 모델을 사용해 문서를 정렬하고 상위 N개를 선택하는 LangChain 구성요소</strong>입니다. 앞 절에서 만든 `cross_encoder`를 `model`에 전달합니다.

| 구분 | 입력 | 하는 일 | 반환값 |
|---|---|---|---|
| 2절 `HuggingFaceCrossEncoder.score` | 질문·본문 쌍 목록 | 각 쌍의 관련성 점수 계산 | 입력 순서의 점수 목록 |
| 3절 `CrossEncoderReranker.compress_documents` | Document 목록과 질문 | 모델로 점수 계산 후 정렬·상위 N개 선택 | 점수순 Document 목록 |

2절에서는 점수를 받아 직접 표로 정렬했습니다. 3절에서는 문서 선택까지 이 구성요소에 맡깁니다.

`CrossEncoderReranker`는 후보를 점수순으로 정렬하고 `top_n`개를 반환합니다. `compress_documents(문서목록, 질문)`은 <strong>준비한 후보를 직접 처리</strong>합니다. 이 단계에서는 본문을 바꾸지 않습니다.

| API·인자 | 의미 |
|---|---|
| `model` | 앞서 한 번 준비한 크로스인코더 |
| `top_n` | 최종 남길 최대 문서 수 |
| `compress_documents` | 주어진 후보에만 후처리 적용 |

<img src="images/lesson01_section03_reranker.png" width="1100" alt="CrossEncoderReranker는 점수 모델을 호출하고, 점수순 정렬과 상위 문서 선택까지 수행합니다.">

CrossEncoderReranker는 점수 모델을 호출하고, 점수순 정렬과 상위 문서 선택까지 수행합니다.


In [ ]:
question = "에이전트가 도구를 잘못 고르거나 호출에 실패할 때 실행 기록을 추적해 원인을 찾는 방법을 배울 책을 찾아 주세요."

# 사용할 점수 모델과 최종 문서 수를 설정합니다.
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)

# 저장된 후보를 점수화하고 상위 3개 문서를 선택합니다.
reranked = list(reranker.compress_documents(candidates, question))

# 선택된 원문을 확인합니다.
show_results(reranked)


In [ ]:
def rank_rows(before, after):
    """원문 ID별 검색 순서와 후처리 순서를 대조하는 행 목록을 반환합니다."""
    # 같은 원문 ID로 비교해야 본문이 짧아져도 출처를 연결할 수 있습니다.
    after_ranks = {}
    for rank, doc in enumerate(after, start=1):
        source_id = doc.metadata["source_id"]
        after_ranks[source_id] = rank

    rows = []
    for rank, doc in enumerate(before, start=1):
        source_id = doc.metadata["source_id"]
        rows.append({"source_id": source_id, "before_rank": rank,
                     "after_rank": after_ranks.get(source_id)})
    return rows


In [ ]:
# 같은 최종 개수로 비교하며, 빠진 문서는 after_rank가 비어 보입니다.
display(pd.DataFrame(rank_rows(candidates, reranked)))
print("리랭킹 전 상위 문서")
show_results(initial_top)
print("리랭킹 후 상위 문서")
show_results(reranked)


<strong>확인할 점:</strong> 리랭킹 전 상위 3개에 질문의 낱말은 많이 겹치지만 조건 하나를 채우지 못하는 책이 있는지 봅니다. 예를 들어 『서버 로그로 추적하는 장애 원인』은 언어 모델 에이전트를 다루지 않고, 『도구 호출 에이전트 첫걸음』은 실행 기록 추적을 다루지 않는다고 밝힙니다.

리랭킹 뒤에 그런 책이 빠졌는지, 새로 들어온 책이 두 조건을 모두 다루는지 원문으로 확인합니다. 순서는 임베딩 결과와 모델에 따라 달라질 수 있으므로 표의 ID를 원문과 대조해 판단합니다.


### 🖐️ 함께 따라하기: PDF의 전후 순서와 근거를 대조합니다

앞에서 보관한 `practice_candidates`를 같은 `reranker`로 처리하세요. 순위 표와 원문 출력 코드는 제공되어 있으므로 리랭킹 호출에 집중합니다.


#### 1) 같은 후보를 리랭킹

`practice_reranked`에 같은 후보와 질문을 리랭커로 처리한 문서 목록을 저장하세요.


In [ ]:
# [작성] ... 부분에 핵심 코드를 작성하세요.
practice_question = "고객사와 정기 회의가 있는 날에도 재택근무를 할 수 있나요?"

# 동일 후보·질문·최종 개수에서 순서와 실제 근거를 비교합니다.
practice_reranked = ...


#### 2) 전후 순위와 원문 확인

제공 셀을 실행해 고객사 미팅 조건을 다루는 원문이 상위 3개에 들어왔는지 판단하세요.


In [ ]:
# [제공 코드]

# 원문 ID로 전후 순위를 맞춰 비교합니다.
display(pd.DataFrame(rank_rows(practice_candidates, practice_reranked)))
print("리랭킹 전 상위 문서")
show_results(practice_initial_top)
print("리랭킹 후 상위 문서")
show_results(practice_reranked)


## 4. ColBERT 방식으로 같은 후보를 재정렬합니다

앞에서는 Cross-Encoder가 질문과 문서를 함께 읽었습니다. 이번에는 질문과 문서를 <strong>각각 토큰별 벡터로 만든 뒤 비교</strong>합니다. BM25·Dense가 찾은 `candidates` 전체를 그대로 사용합니다.

실습 모델은 다국어·MIT 라이선스의 `BAAI/bge-m3`입니다. 이 모델의 <strong>ColBERT 방식 다중 벡터 기능</strong>을 사용합니다. 원래 ColBERT 논문의 체크포인트와는 구분합니다. 기존 하이브리드 검색기의 임베딩은 그대로 사용합니다.

`return_colbert_vecs=True`는 토큰별 벡터를 반환하라는 설정입니다. 질문 토큰마다 가장 잘 맞는 문서 토큰을 찾는 MaxSim으로 점수를 계산합니다. BGE-M3는 이 합을 질문 토큰 수로 나눈 평균을 사용합니다. 같은 질문에서는 합산 방식과 후보 순위가 같습니다.

[공식 모델 카드](https://huggingface.co/BAAI/bge-m3) · [공식 점수 구현](https://github.com/FlagOpen/FlagEmbedding/blob/v1.4.2/FlagEmbedding/inference/embedder/encoder_only/m3.py)

`FlagEmbedding==1.4.2`가 필요합니다. 최초 실행에는 별도 모델 가중치 약 2.3GB를 내려받습니다. 아래에서는 CPU로 실행합니다.

<img src="images/lesson01_section04_colbert.png" width="1100" alt="질문과 문서를 각각 토큰별 벡터로 만든 뒤, 질문 토큰마다 가장 잘 맞는 문서 토큰을 찾아 점수를 계산합니다.">

질문과 문서를 각각 토큰별 벡터로 만든 뒤, 질문 토큰마다 가장 잘 맞는 문서 토큰을 찾아 점수를 계산합니다.


In [ ]:
# ====== ColBERT 방식의 토큰 벡터 모델 준비 ======
from FlagEmbedding import BGEM3FlagModel

colbert_model = BGEM3FlagModel(
    "BAAI/bge-m3",            # 사용할 다국어 모델
    devices="cpu",            # CPU에서 추론
    batch_size=2,              # 한 번에 인코딩할 텍스트 수
    passage_max_length=1024,   # 문서의 최대 토큰 수
    return_dense=False,       # 단일 벡터 출력 안 함
    return_colbert_vecs=True,  # 토큰별 벡터 출력
)


#### 1) 질문을 인코딩하고 반환값을 확인합니다

`encode_queries`에는 질문을 목록으로 전달합니다. 반환 딕셔너리의 각 키는 다음 값을 담습니다.

| 반환 키 | 값의 의미 |
|---|---|
| `dense_vecs` | 텍스트마다 전체 의미를 담은 벡터 하나 |
| `lexical_weights` | 텍스트마다 토큰 ID와 중요도 점수를 담은 딕셔너리 |
| `colbert_vecs` | 텍스트마다 토큰별 벡터를 모은 배열 `(토큰 수, 벡터 차원)` |

이번 설정에서는 `dense_vecs`와 `lexical_weights`의 값은 `None`이고, `colbert_vecs`만 사용합니다. `colbert_vecs[0]`에서 첫 질문의 토큰별 벡터를 꺼냅니다.


In [ ]:
question = "에이전트가 도구를 잘못 고르거나 호출에 실패할 때 실행 기록을 추적해 원인을 찾는 방법을 배울 책을 찾아 주세요."

# 질문 목록을 토큰별 벡터로 바꿉니다.
query_output = colbert_model.encode_queries([question])

# 질문을 하나 넣었으므로 첫 번째 결과를 꺼냅니다.
print("반환 키:", list(query_output.keys()))
query_vectors = query_output["colbert_vecs"][0]

print("질문 벡터 모양(토큰 수, 벡터 차원):", query_vectors.shape)


#### 2) 후보 문서를 인코딩하고 반환값을 확인합니다

`encode_corpus`에는 문서 본문 목록을 전달합니다. 각 문서에 토큰별 벡터가 하나씩 대응합니다.


In [ ]:
# 모든 후보 본문을 검색 당시 순서대로 준비합니다.
candidate_texts = [doc.page_content for doc in candidates]

document_output = colbert_model.encode_corpus(candidate_texts)

# colbert_vecs의 각 항목이 문서 하나의 토큰별 벡터입니다.
document_vectors = document_output["colbert_vecs"]

print("반환 키:", list(document_output.keys()))
print("벡터를 만든 문서 수:", len(document_vectors))
for doc, vectors in zip(candidates, document_vectors):
    print("원문 ID:", doc.metadata["source_id"], "벡터 모양:", vectors.shape)


#### 3) 문서 하나의 관련성 점수를 확인합니다

`colbert_score(질문 벡터, 문서 벡터)`로 두 토큰 벡터를 비교합니다. 먼저 첫 후보 하나로 호출 방법을 확인합니다.


In [ ]:
# 첫 문서 한 개로 토큰 매칭 점수의 반환값을 확인합니다.
first_score = colbert_model.colbert_score(query_vectors, document_vectors[0])

print("첫 후보 ID:", candidates[0].metadata["source_id"])
print("첫 후보의 ColBERT 점수:", float(first_score))


#### 4) 전체 후보를 점수화하고 상위 문서를 선택합니다

앞 셀의 호출을 모든 후보에 반복합니다. 점수가 높은 후보의 위치로 원문을 선택합니다.


In [ ]:
# 모든 후보의 관련성 점수를 차례로 계산합니다.
colbert_scores = []
for vectors in document_vectors:
    score = colbert_model.colbert_score(query_vectors, vectors)
    colbert_scores.append(float(score))

# 점수순 후보 위치로 상위 3개 원문을 선택합니다.
colbert_order = sorted(
    range(len(candidates)), key=lambda index: colbert_scores[index], reverse=True,
)
colbert_reranked = [candidates[index] for index in colbert_order[:3]]

print("ColBERT 점수(후보 순서):", colbert_scores)


#### 5) ColBERT 순위 비교표를 만듭니다

같은 후보의 원래 순위와 ColBERT 점수에 따른 순위를 표에 담습니다.


In [ ]:
# 같은 위치의 후보·토큰 벡터·점수를 묶어 한 행에 기록합니다.
colbert_rows = []
for before_rank, (doc, vectors, score) in enumerate(zip(candidates, document_vectors, colbert_scores), start=1):
    colbert_rows.append({
        "before_rank": before_rank,
        "question": question, "source_id": doc.metadata["source_id"],
        "title": doc.metadata["title"], "text": doc.page_content, "token_count": len(vectors), "score": score,
    })
# 검색 당시 순위는 유지하고, 별도 표를 점수순으로 정렬합니다.
colbert_table = pd.DataFrame(colbert_rows)
# 점수가 같으면 검색 당시 순서를 유지합니다.
ranked_colbert_table = colbert_table.sort_values("score", ascending=False, kind="stable")
ranked_colbert_table = ranked_colbert_table.reset_index(drop=True)
ranked_colbert_table["after_rank"] = range(1, len(ranked_colbert_table) + 1)


#### 6) 순위와 선택한 원문을 출력합니다

순위 비교표와 최종 선택 문서를 읽습니다. 출력 셀만 다시 실행해도 모델은 다시 호출하지 않습니다.


In [ ]:
# 저장된 표를 출력하므로 이 셀은 모델을 다시 호출하지 않습니다.
print("검색 당시 순서(before_rank)")
display(colbert_table[["before_rank", "source_id", "title", "score"]])
print("점수에 따른 순서(after_rank): before_rank와 비교하세요.")
# 같은 행의 두 순위와 원문 ID를 보고 어떤 문서가 이동했는지 확인합니다.
display(ranked_colbert_table[["before_rank", "after_rank", "source_id", "title", "score", "token_count"]])
show_results(colbert_reranked)


<strong>확인할 점:</strong> `before_rank`와 `after_rank`로 후보 전체의 순위 변화를 먼저 확인하고, 선택한 원문이 질문에 답하는지 읽습니다. ColBERT 방식은 문서의 토큰 벡터를 보관해 재사용할 수 있으며, 문서마다 벡터 하나를 저장할 때보다 저장량이 늘어납니다.


### 🖐️ 함께 따라하기: ColBERT 방식으로 PDF 후보를 재정렬합니다

같은 PDF 후보 전체를 ColBERT 방식으로 처리하는 핵심 호출을 작성합니다. 점수 정렬·표 생성·원문 출력은 제공 코드를 실행해 확인하세요.


#### 1) 질문 인코딩

`encode_queries` 호출을 작성하세요.


In [ ]:
# [작성] ... 부분에 핵심 코드를 작성하세요.
practice_question = "고객사와 정기 회의가 있는 날에도 재택근무를 할 수 있나요?"

# 질문 목록을 모델에 전달하는 호출을 작성합니다.
practice_query_output = ...


#### 2) 질문 벡터 확인

제공 셀로 반환 키와 토큰 벡터 모양을 확인하세요.


In [ ]:
# [제공 코드]

# 질문 하나의 토큰별 벡터를 꺼내 구조를 확인합니다.
print("반환 키:", list(practice_query_output.keys()))
practice_query_vectors = practice_query_output["colbert_vecs"][0]

print("질문 벡터 모양:", practice_query_vectors.shape)


#### 3) 문서 인코딩

`encode_corpus` 호출을 작성하세요.


In [ ]:
# [작성] ... 부분에 핵심 코드를 작성하세요.

# 검색 당시 순서의 PDF 본문 목록은 제공되어 있습니다.
practice_texts = [doc.page_content for doc in practice_candidates]

# 문서 본문 목록을 인코딩하는 호출을 작성합니다.
practice_document_output = ...


#### 4) 문서 벡터 확인

제공 셀로 문서마다 생성된 토큰 벡터를 확인하세요.


In [ ]:
# [제공 코드]

# 문서별 토큰 벡터를 보관하고 원문 ID와 대응시킵니다.
practice_document_vectors = practice_document_output["colbert_vecs"]

print("반환 키:", list(practice_document_output.keys()))
for doc, vectors in zip(practice_candidates, practice_document_vectors):
    print("원문 ID:", doc.metadata["source_id"], "벡터 모양:", vectors.shape)


#### 5) 토큰 매칭 점수 계산

반복문 안의 `colbert_score` 호출을 채우세요. 점수순 정렬과 상위 3개 선택은 제공되어 있습니다.


In [ ]:
# [작성] ... 부분에 핵심 코드를 작성하세요.

# 모든 후보의 토큰 매칭 점수를 구하는 호출을 작성합니다.
practice_colbert_scores = []
for vectors in practice_document_vectors:
    score = ...
    practice_colbert_scores.append(float(score))

# 점수순 후보 위치로 상위 3개 원문을 선택합니다.
practice_colbert_order = sorted(
    range(len(practice_candidates)),
    key=lambda index: practice_colbert_scores[index], reverse=True,
)
practice_colbert_reranked = [practice_candidates[index] for index in practice_colbert_order[:3]]


#### 6) 점수 확인과 비교표 준비

제공 셀을 실행해 점수와 전후 순위 표를 준비하세요.


In [ ]:
# [제공 코드]

# 전체 후보의 점수 목록을 먼저 확인합니다.
print("PDF ColBERT 점수(후보 순서):", practice_colbert_scores)

# 후보별 질문·원문·점수를 함께 읽고 순위를 해석합니다.
practice_colbert_rows = []
for before_rank, (doc, score) in enumerate(zip(practice_candidates, practice_colbert_scores), start=1):
    practice_colbert_rows.append({
        "before_rank": before_rank,
        "question": practice_question, "source_id": doc.metadata["source_id"],
        "title": doc.metadata["title"], "text": doc.page_content, "score": score,
    })
# 검색 당시 순위는 유지하고, 별도 표를 점수순으로 정렬합니다.
practice_colbert_table = pd.DataFrame(practice_colbert_rows)
# 점수가 같으면 검색 당시 순서를 유지합니다.
practice_ranked_colbert_table = practice_colbert_table.sort_values("score", ascending=False, kind="stable")
practice_ranked_colbert_table = practice_ranked_colbert_table.reset_index(drop=True)
practice_ranked_colbert_table["after_rank"] = range(1, len(practice_ranked_colbert_table) + 1)


#### 7) 순위와 선택한 원문 비교

제공 셀로 전후 순위와 ColBERT가 선택한 원문을 확인하세요.


In [ ]:
# [제공 코드]

# 저장된 표를 출력하므로 이 셀은 모델을 다시 호출하지 않습니다.
print("검색 당시 순서(before_rank)")
display(practice_colbert_table[["before_rank", "source_id", "title", "score"]])
print("점수에 따른 순서(after_rank): before_rank와 비교하세요.")
# 같은 행의 두 순위와 원문 ID를 보고 어떤 문서가 이동했는지 확인합니다.
display(practice_ranked_colbert_table[["before_rank", "after_rank", "source_id", "title", "score"]])
print("ColBERT 방식이 선택한 원문")
show_results(practice_colbert_reranked)


## 5. 검색과 리랭킹을 연결합니다

`ContextualCompressionRetriever`는 `base_retriever`로 검색하고 `base_compressor`로 후처리합니다. 여기에는 본문 추출기뿐 아니라 리랭커도 연결할 수 있습니다.

반복해서 사용할 검색 기능을 묶는 데 편리합니다. 다만 `.invoke()`할 때마다 1차 검색부터 다시 수행하므로, <strong>같은 후보를 비교할 때는 앞 절처럼 직접 후처리</strong>합니다.

<img src="images/lesson01_section05_retriever.png" width="1100" alt="ContextualCompressionRetriever의 invoke는 질문으로 다시 검색한 뒤, 리랭커가 고른 문서를 반환합니다.">

ContextualCompressionRetriever의 invoke는 질문으로 다시 검색한 뒤, 리랭커가 고른 문서를 반환합니다.


In [ ]:
# 새 질문에는 검색과 리랭킹을 한 번에 적용합니다.
ranked_retriever = ContextualCompressionRetriever(base_retriever=hybrid, base_compressor=reranker)
new_question = "사내 문서를 검색하는 도구를 붙여 근거와 함께 답하는 RAG 에이전트를 만드는 방법을 설명하는 책을 찾아 주세요."
new_results = ranked_retriever.invoke(new_question)

show_results(new_results)


결과로 받은 책이 <strong>사내 문서 검색을 도구로 만들어 에이전트에 연결하고, 답변에 근거 문서를 밝히는지</strong> 원문으로 읽습니다.

『RAG 파이프라인 입문』처럼 검색과 답변을 고정된 체인으로만 잇는 책이나 『웹 검색 도구로 답하는 에이전트』처럼 사내 문서를 다루지 않는 책이 남았다면 원인을 나눠 봅니다. 연결된 검색기는 후보를 보여 주지 않으므로 `hybrid.invoke(new_question)`로 후보를 따로 확인합니다. 후보에 조건을 모두 채우는 책이 세 권 이상 있었다면 리랭커가 조건 하나를 놓친 것이고, 그보다 적었다면 검색 단계의 한계입니다.


### 🖐️ 함께 따라하기: 새 PDF 질문을 연결된 검색기로 처리합니다

`practice_hybrid`와 `reranker`를 연결해 새로운 보안 질문을 처리합니다. 연결과 호출만 작성하고, 결과는 제공된 출력 셀로 확인하세요.


#### 1) 검색기 연결과 새 질문 실행

`practice_ranked_retriever`를 만들고 `practice_security_results`에 새 질문의 결과를 저장하세요.


In [ ]:
# [작성] ... 부분에 핵심 코드를 작성하세요.
# 새로운 질문을 처리할 때 편의 검색기를 사용합니다.
practice_ranked_retriever = ...
practice_security_question = "보안 문제가 생기면 직원의 재택근무를 중단시킬 수 있나요?"
practice_security_results = ...


#### 2) 새 질문의 결과 확인

제공 셀을 실행해 보안 문제로 재택근무를 해제하는 기준과 절차가 있는지 확인하세요.


In [ ]:
# [제공 코드]

# 새 질문으로 검색·리랭킹한 원문을 읽습니다.
show_results(practice_security_results)


## 이번 강의 정리

| 구분 | 확인 기준 |
|---|---|
| 후보 검색 | 필요한 원문이 후보 안에 있는가 |
| 리랭킹 | 같은 후보·최종 개수에서 근거 순서가 나아졌는가 |
| ColBERT 방식 | 토큰별 벡터로 같은 후보 전체를 점수화하고 원문을 선택했는가 |

## ⏭️ 다음 시간 예고

선택한 문서에서 질문에 필요한 내용을 추출합니다.
